In [1]:
# =========================
# 1) Imports
# =========================
import json
import re
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score
)

# =========================
# 2) Robust loader for messy JSON / JSONL / concatenated JSON objects
# =========================
def load_messy_json(path):
    """
    Tries to recover dict records from:
    - valid JSON list
    - valid JSON object
    - JSONL
    - concatenated JSON objects
    - partially messy files with extra non-JSON text
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()

    content = content.replace("\ufeff", "").strip()
    decoder = json.JSONDecoder()
    objects = []
    i = 0
    n = len(content)

    while i < n:
        # move to next possible JSON start
        while i < n and content[i] not in "{[":
            i += 1
        if i >= n:
            break

        try:
            obj, end = decoder.raw_decode(content, i)

            if isinstance(obj, dict):
                objects.append(obj)
            elif isinstance(obj, list):
                for item in obj:
                    if isinstance(item, dict):
                        objects.append(item)

            i = end
        except json.JSONDecodeError:
            i += 1

    df = pd.DataFrame(objects)

    # Keep only rows that look like payload records
    if not df.empty:
        possible_cols = [c for c in df.columns if isinstance(c, str)]
        df.columns = [c.strip().lower() if isinstance(c, str) else c for c in df.columns]

        required_like = {"payload", "type", "severity", "description", "context"}
        if len(set(df.columns) & required_like) == 0:
            print("Warning: parsed objects, but they do not look like the expected dataset structure.")

    return df


# =========================
# 3) Load dataset
# =========================
path = "/kaggle/input/datasets/cyberprince/web-application-payloads-dataset/WEB_APPLICATION_PAYLOADS.jsonl"
df = load_messy_json(path)

print("Initial shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

# =========================
# 4) Clean column names and choose target
# =========================
df.columns = [c.strip().lower() if isinstance(c, str) else c for c in df.columns]

# Choose the target column:
# For your project, type is usually the better target (SQLi, XSS, CSRF, SSRF, CMDi)
if "type" in df.columns:
    label_col = "type"
elif "severity" in df.columns:
    label_col = "severity"
else:
    raise ValueError("No suitable target column found. Expected 'type' or 'severity'.")

print("Target column:", label_col)

# =========================
# 5) Basic inspection
# =========================
if "id" in df.columns:
    df = df.drop_duplicates(subset=["id"])
elif "payload" in df.columns:
    df = df.drop_duplicates(subset=["payload"])
else:
    df = df.drop_duplicates()

print("\nShape after dropping duplicates:", df.shape)

print("\nMissing values per column:")
print(df.isna().sum())

print(f"\nTarget distribution ({label_col}):")
print(df[label_col].value_counts(dropna=False))

# =========================
# 6) Build text input
# =========================
def clean_text(x):
    x = "" if pd.isna(x) else str(x)
    x = x.replace("\n", " ").replace("\t", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip().lower()

text_cols = []
for col in ["description", "payload", "context", "example_query", "example_usage"]:
    if col in df.columns:
        text_cols.append(col)

if "payload" not in df.columns:
    raise ValueError("No 'payload' column found. This dataset must contain payload text.")

# Create combined text feature
for col in text_cols:
    df[col] = df[col].fillna("").map(clean_text)

df["text"] = df[text_cols].agg(" ".join, axis=1).map(clean_text)

# Remove empty rows
df = df[df["text"].str.len() > 0].copy()
df = df[df[label_col].notna()].copy()
df[label_col] = df[label_col].astype(str).str.strip().str.lower()

print("\nFinal shape after cleaning:", df.shape)
print("\nSample processed rows:")
print(df[["text", label_col]].head())

# =========================
# 7) Train/test split
# =========================
X = df["text"]
y = df[label_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

# =========================
# 8) Models
#    Char n-grams work well for payload strings
# =========================
def make_tfidf():
    return TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        max_features=30000,
        lowercase=True
    )

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", make_tfidf()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
    ]),
    
    "Linear SVM": Pipeline([
        ("tfidf", make_tfidf()),
        ("clf", LinearSVC(class_weight="balanced"))
    ]),
    
    "Random Forest": Pipeline([
        ("tfidf", make_tfidf()),
        ("svd", TruncatedSVD(n_components=100, random_state=42)),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        ))
    ])
}

# =========================
# 9) Cross-validation + test evaluation
# =========================
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "accuracy": "accuracy",
        "f1_macro": "f1_macro",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "balanced_accuracy": "balanced_accuracy"
    }

    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    print(f"CV Accuracy: {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")
    print(f"CV F1 Macro: {cv_results['test_f1_macro'].mean():.4f} ± {cv_results['test_f1_macro'].std():.4f}")
    print(f"CV Precision Macro: {cv_results['test_precision_macro'].mean():.4f} ± {cv_results['test_precision_macro'].std():.4f}")
    print(f"CV Recall Macro: {cv_results['test_recall_macro'].mean():.4f} ± {cv_results['test_recall_macro'].std():.4f}")
    print(f"CV Balanced Accuracy: {cv_results['test_balanced_accuracy'].mean():.4f} ± {cv_results['test_balanced_accuracy'].std():.4f}")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
    print("Test Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
    print("Test F1 Macro:", f1_score(y_test, y_pred, average="macro"))
    print("Test Precision Macro:", precision_score(y_test, y_pred, average="macro", zero_division=0))
    print("Test Recall Macro:", recall_score(y_test, y_pred, average="macro", zero_division=0))

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred, zero_division=0))

    print("Confusion Matrix:\n")
    print(confusion_matrix(y_test, y_pred))

    return {
        "model": model,
        "cv_accuracy_mean": cv_results["test_accuracy"].mean(),
        "cv_f1_macro_mean": cv_results["test_f1_macro"].mean(),
        "cv_precision_macro_mean": cv_results["test_precision_macro"].mean(),
        "cv_recall_macro_mean": cv_results["test_recall_macro"].mean(),
        "cv_balanced_accuracy_mean": cv_results["test_balanced_accuracy"].mean(),
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1_macro": f1_score(y_test, y_pred, average="macro"),
        "test_precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "test_recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "y_pred": y_pred
    }

results = {}

for name, model in models.items():
    results[name] = evaluate_model(name, model, X_train, y_train, X_test, y_test)

# =========================
# 10) Comparison table
# =========================
comparison = pd.DataFrame([
    {
        "Model": name,
        "CV Accuracy": info["cv_accuracy_mean"],
        "CV F1 Macro": info["cv_f1_macro_mean"],
        "CV Precision Macro": info["cv_precision_macro_mean"],
        "CV Recall Macro": info["cv_recall_macro_mean"],
        "CV Balanced Acc": info["cv_balanced_accuracy_mean"],
        "Test Accuracy": info["test_accuracy"],
        "Test F1 Macro": info["test_f1_macro"],
        "Test Precision Macro": info["test_precision_macro"],
        "Test Recall Macro": info["test_recall_macro"],
        "Test Balanced Acc": info["test_balanced_accuracy"]
    }
    for name, info in results.items()
]).sort_values(by="CV F1 Macro", ascending=False)

print("\n\nModel comparison:\n")
print(comparison)

# =========================
# 11) Pick best model and save it
# =========================
best_model_name = comparison.iloc[0]["Model"]
best_model = results[best_model_name]["model"]

print("\nBest model:", best_model_name)

joblib.dump(
    {
        "model": best_model,
        "label_col": label_col,
        "text_columns": text_cols
    },
    "best_payload_classifier.pkl"
)

print("Saved as best_payload_classifier.pkl")

# =========================
# 12) Load later and predict
# =========================
bundle = joblib.load("best_payload_classifier.pkl")
loaded_model = bundle["model"]

sample_payload = "' OR '1'='1"
sample_context = "Login username"

sample_text = clean_text(sample_payload + " " + sample_context)
pred = loaded_model.predict([sample_text])[0]

print("\nSample prediction:", pred)

Initial shape: (480, 8)
Columns: ['id', 'description', 'payload', 'context', 'type', 'severity', 'example_query', 'example_usage']
         id                                        description  \
0  sqli-001                Basic tautology-based SQL injection   
1  sqli-002          Union-based SQL injection to extract data   
2  sqli-003                Blind SQL injection with time delay   
3  sqli-004  Error-based SQL injection to reveal database v...   
4  sqli-005                  Boolean-based blind SQL injection   

                                          payload                    context  \
0                                     ' OR '1'='1  Login form username input   
1  ' UNION SELECT username, password FROM users--         Search input field   
2                      '; WAITFOR DELAY '0:0:5'--        ID parameter in URL   
3                ' AND 1=CONVERT(int,@@version)--              User ID input   
4                                     ' AND 1=1--               Search f

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


CV Accuracy: 0.9896 ± 0.0097
CV F1 Macro: 0.9836 ± 0.0183
CV Precision Macro: 0.9884 ± 0.0113
CV Recall Macro: 0.9845 ± 0.0177
CV Balanced Accuracy: 0.9845 ± 0.0177

Test Accuracy: 0.96875
Test Balanced Accuracy: 0.982166159199652
Test F1 Macro: 0.9812903470798208
Test Precision Macro: 0.9822510822510823
Test Recall Macro: 0.982166159199652

Classification Report:

                   precision    recall  f1-score   support

       blind-time       1.00      1.00      1.00         6
    boolean-blind       1.00      1.00      1.00         2
command injection       1.00      1.00      1.00        19
             csrf       1.00      0.89      0.94        19
      error-based       1.00      1.00      1.00         1
        reflected       1.00      0.91      0.95        11
             ssrf       0.90      1.00      0.95        19
  stacked-queries       1.00      1.00      1.00         2
           stored       0.90      1.00      0.95         9
        tautology       1.00      1.00   

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


CV Accuracy: 0.9974 ± 0.0052
CV F1 Macro: 0.9976 ± 0.0047
CV Precision Macro: 0.9982 ± 0.0036
CV Recall Macro: 0.9974 ± 0.0052
CV Balanced Accuracy: 0.9974 ± 0.0052

Test Accuracy: 0.9895833333333334
Test Balanced Accuracy: 0.9952153110047847
Test F1 Macro: 0.9952119952119951
Test Precision Macro: 0.9954545454545454
Test Recall Macro: 0.9952153110047847

Classification Report:

                   precision    recall  f1-score   support

       blind-time       1.00      1.00      1.00         6
    boolean-blind       1.00      1.00      1.00         2
command injection       1.00      1.00      1.00        19
             csrf       1.00      0.95      0.97        19
      error-based       1.00      1.00      1.00         1
        reflected       1.00      1.00      1.00        11
             ssrf       0.95      1.00      0.97        19
  stacked-queries       1.00      1.00      1.00         2
           stored       1.00      1.00      1.00         9
        tautology       1.00

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predic

CV Accuracy: 0.9766 ± 0.0151
CV F1 Macro: 0.9285 ± 0.0512
CV Precision Macro: 0.9269 ± 0.0521
CV Recall Macro: 0.9333 ± 0.0476
CV Balanced Accuracy: 0.9333 ± 0.0476

Test Accuracy: 0.9479166666666666
Test Balanced Accuracy: 0.8829926054806438
Test F1 Macro: 0.8811924189331122
Test Precision Macro: 0.882552800734619
Test Recall Macro: 0.8829926054806438

Classification Report:

                   precision    recall  f1-score   support

       blind-time       1.00      1.00      1.00         6
    boolean-blind       1.00      1.00      1.00         2
command injection       1.00      1.00      1.00        19
             csrf       0.94      0.89      0.92        19
      error-based       0.00      0.00      0.00         1
        reflected       1.00      0.82      0.90        11
             ssrf       0.86      1.00      0.93        19
  stacked-queries       1.00      1.00      1.00         2
           stored       0.90      1.00      0.95         9
        tautology       1.00 